# VD19 — AM-T-HAR-Blend variance vs 0DTE implied volatility

Protocol: `discussions/Volatility_Dynamics/037_claude_vd19_model_vs_implied_preregistration.md`.

At each entry (04:00 UTC primary, 240 min to expiry; 06:00 → 120 min; 12:00 → 1200 min), compare the model's forecast s.d. of the log move to expiry with the ATM straddle's implied s.d.:

$$x=\log\frac{\sigma_{model}}{\sigma_{imp}},\qquad y_{rv}=\log\frac{RV_{(t,T]}}{\sigma_{imp}^2},\qquad y_{rv}=a+\beta x+\varepsilon.$$

$\beta>0$ (session-bootstrap CI) means the model knows something the option price does not. The economics are the straddle's $R_{net}$ at ask + fee when $\sigma_{model}\ge1.25\,\sigma_{imp}$.

In [ ]:
import os,sys,json,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as v14, markov_har_variance as v18
from data_engineering.modeling import vrp_long_straddle as ls, model_vs_implied as mv
for m in (v14,v18,ls,mv):importlib.reload(m)
assert mv.VERSION.startswith('VD19-0.1.1')
P=mv.Params()
SYNTHETIC=os.getenv('VD19_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='vd19-');OUT=Path(tmp.name)/'19'
else:
    OUT=ROOT/'notebooks'/'Volatility_Dynamics'/'19'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_model_vs_implied.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-1500:]);print(chk.stderr[-800:]);assert chk.returncode==0,'VD19 tests failed'
print('VD19',mv.VERSION,'synthetic:',SYNTHETIC,'output:',OUT)

## 1. Model: VD18 HAR_4 refit on the VD18 sample, forecasts at collector-era entries

In [ ]:
if not SYNTHETIC:
    HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
    H=sorted(set(mv.ENTRY_HORIZON.values()))
    raw_fit,excl=v14.load_returns(HIST,mv.FIT_ASSETS,end=pd.Timestamp(P.fit_sample_end))
    frame=mv.build_frame(raw_fit,H,v14,v18)
    FITS=mv.fit_har_models(frame,H,v18)
    print('HAR_4 fitted at',H,'| excluded:',excl,'| fit sample',frame.state_time.min(),'->',frame.state_time.max())
    del frame,raw_fit
    raw_all,_=v14.load_returns(HIST,mv.TEST_ASSETS,end=pd.Timestamp(P.forecast_end),min_days=365)
    PANEL=mv.forecast_panel(raw_all,v14,v18)
    print('forecast panel through',PANEL.state_time.max())

## 2. Options: ATM straddles at the registered entries (VRP 02 code), settlement and R_net

In [ ]:
if SYNTHETIC:
    d,sm,rv=mv.synthetic_comparison(n_sessions=25,seed=5,model_skill=0.6)
    d['entry_hour']=4;d['h']=240
    D=mv.comparison_frame(d,sm,rv)
else:
    import duckdb
    DATA_ROOT=Path(os.environ['DERIBIT_DATA_ROOT'])
    con=duckdb.connect(str(DATA_ROOT/'database'/'research.duckdb'),read_only=True)
    con.execute("SET TimeZone='UTC'")   # v0.1.1: EXTRACT(hour) must be UTC
    try:
        have=set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
        cols=[c for c in ls.CHAIN_COLUMNS if c!='settlement_currency']+(['settlement_currency'] if 'settlement_currency' in have else [])
        chain=con.execute(f'''
            WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
            SELECT {', '.join('n.'+c for c in cols)}
            FROM normalized_option_chain n
            JOIN prim p ON p.state_id=n.state_id AND p.option_family_id=n.option_family_id
            WHERE n.minutes_to_expiry BETWEEN {ls.MIN_TTE_MIN} AND {ls.MAX_TTE_MIN}
              AND EXTRACT(minute FROM n.state_time)=0
              AND EXTRACT(hour FROM n.state_time) IN ({','.join(str(h) for h in mv.ENTRY_HORIZON)})
              AND n.state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}'
              AND abs(ln(n.strike/n.option_forward_price)) < 0.05
              AND n.underlying IN ({','.join(repr(a) for a in mv.TEST_ASSETS)})''').df()
        idx=con.execute(f'''SELECT state_time, underlying, futures_index_price FROM time_series_feature_state
                            WHERE state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}' ''').df()
    finally:
        con.close()
    if 'settlement_currency' not in chain:chain['settlement_currency']=None
    st=ls.select_straddles(ls.prepare_legs(chain))
    E=mv.entries(st,P)
    sessions=sorted(E.session.unique())
    approx=ls.approx_settlement(idx,sessions)
    names=sorted({ls.index_name(u,c) for u,c in zip(E.underlying,E.settlement_currency)})
    cache=OUT/'19_delivery_prices.csv'
    delivery=(pd.read_csv(cache,dtype={'session':str}) if (os.getenv('VD19_OFFLINE','')=='1' and cache.exists())
              else ls.fetch_delivery_prices(names,cache_csv=cache))
    delivery=delivery[delivery.session.isin(sessions)]
    X=ls.straddle_pnl(ls.attach_settlement(E,delivery,approx)).reset_index(drop=True)
    origins=X[['underlying','state_time','h']]
    IPANEL=mv.index_panel(idx)
    D=mv.comparison_frame(X,mv.model_sigma(PANEL,origins,FITS,v18),mv.realised_variance(PANEL,origins),
                          mv.realised_variance(IPANEL,origins))
print('rows:',len(D),'| sessions:',D.session.nunique(),'| by entry:',D.groupby('entry_hour').size().to_dict())
display(D.groupby(['entry_hour','underlying'])[['x','y_rv','y','R_net']].agg(['mean','count']).round(3))

## 3. Information test and decomposition

In [ ]:
res=[]
for hr,g in D.groupby('entry_hour'):
    for ycol,label in (('y_rv','PRIMARY realised path variance / implied' if hr==mv.PRIMARY_ENTRY else 'realised path variance / implied'),
                       ('y_rv_index','DIAGNOSTIC (v0.1.1): Deribit-index path variance / implied'),
                       ('y','settlement move / implied')):
        r=mv.session_bootstrap(g,['x'],ycol,P,seed_offset=hr*10+len(ycol));res.append({'entry_hour':hr,'outcome':ycol,'label':label,**r})
    enc=mv.session_bootstrap(g,['log_sigma_imp','log_sigma_model'],'log_rv',P,seed_offset=hr*10+7)
    res.append({'entry_hour':hr,'outcome':'log_rv ~ imp + model','label':'encompassing',**enc})
RES=pd.DataFrame(res);display(RES.round(4))
print('Mean x by entry (model vs implied level; <0 means implied above model):')
display(D.groupby('entry_hour').x.describe().round(3))

## 4. Economics: straddles bought only when the model is >= 25% above implied

In [ ]:
thr=np.log(P.trade_threshold_ratio)
econ=[]
for hr,g in D.groupby('entry_hour'):
    for name,sub in (('all',g),('x>=log1.25',g[g.x>=thr]),('x<log1.25',g[g.x<thr])):
        econ.append({'entry_hour':hr,'subset':name,**mv.session_mean_boot(sub,'R_net',P)})
ECON=pd.DataFrame(econ);display(ECON.round(4))
terc=D.assign(tercile=D.groupby('entry_hour').x.transform(lambda v:pd.qcut(v,3,labels=['low','mid','high']))).groupby(['entry_hour','tercile'],observed=True).agg(
    n=('R_net','size'),x=('x','mean'),y_rv=('y_rv','mean'),R_net=('R_net','mean'))
print('Descriptive terciles of x (cut-offs from this sample; not a trading rule):');display(terc.round(3))

In [ ]:
prim=RES[(RES.entry_hour==mv.PRIMARY_ENTRY)&(RES.outcome=='y_rv')].iloc[0].to_dict()
sel=ECON[(ECON.entry_hour==mv.PRIMARY_ENTRY)&(ECON.subset=='x>=log1.25')]
dec=mv.decision(prim,sel.iloc[0].to_dict() if len(sel) else {},synthetic=SYNTHETIC)
print(json.dumps(dec,indent=2,default=float))
D[['session','underlying','entry_hour','h','sigma_model','sigma_imp','rv_path','rv_index','x','y_rv','y_rv_index','y','R_net','settle_source']
  if 'settle_source' in D else ['session','underlying','x','y_rv','y','R_net']].to_csv(OUT/'19_rows.csv',index=False,float_format='%.6g')
RES.to_csv(OUT/'19_information.csv',index=False);ECON.to_csv(OUT/'19_economics.csv',index=False)
(OUT/'19_decision.json').write_text(json.dumps(dec,indent=2,default=float),encoding='utf-8')
(OUT/'19_manifest.json').write_text(json.dumps({'registration':mv.registration(P),'synthetic':SYNTHETIC,
    'sessions':[str(D.session.min()),str(D.session.max())],'n_rows':len(D)},indent=2,default=str),encoding='utf-8')
print('saved',sorted(p.name for p in OUT.iterdir()))